# End-to-end PyMetaAnalysis quickstart

This notebook uses a small **synthetic** binary dataset. It demonstrates the public workflow without representing a real clinical question.

In [ ]:
%matplotlib inline

import pandas as pd

import meta_analyze as ma

## Prepare study-level data

The DataFrame index becomes the study label because `study=` is omitted.

In [ ]:
studies = pd.DataFrame(
    {
        "event_treat": [12, 5, 20, 7, 16, 9],
        "n_treat": [100, 80, 120, 90, 110, 95],
        "event_control": [18, 9, 15, 10, 21, 13],
        "n_control": [110, 75, 130, 95, 115, 100],
    },
    index=pd.Index(
        ["Trial A", "Trial B", "Trial C", "Trial D", "Trial E", "Trial F"],
        name="study",
    ),
)
studies

## Fit a random-effects risk-ratio model

The model uses inverse-variance pooling, REML heterogeneity, and the safeguarded Hartung-Knapp mean interval.

In [ ]:
result = ma.meta_binary(
    studies,
    event_treat="event_treat",
    n_treat="n_treat",
    event_control="event_control",
    n_control="n_control",
    measure="RR",
    method="IV",
    model="random",
    tau2_method="REML",
    ci_method="hartung_knapp_adhoc",
)
result.summary()

## Inspect model and display scales

Risk ratios are fitted on the log scale. Display properties expose exponentiated values while the study table retains auditable model-scale quantities.

In [ ]:
print("log pooled effect:", result.estimate)
print("pooled risk ratio:", result.display_estimate)
print("risk-ratio CI:", result.display_ci)
print("risk-ratio prediction interval:", result.display_prediction_interval)
print("I-squared proportion:", result.i2)

result.study_results[["study", "effect", "variance", "included", "normalized_weight"]]

## Generate reporting material

Generated Methods text is a starting point for human review. The structured report records resolved methods, row decisions, diagnostics, and provenance.

In [ ]:
print(result.method_details())

report = result.report(include_studies=True)
payload = report.to_dict()
sorted(payload)

## Check sensitivity and draw a forest plot

In [ ]:
leave_one_out = result.leave_one_out().to_dataframe()
assert len(leave_one_out) == result.k
leave_one_out[
    ["omitted_study", "display_estimate", "display_ci_low", "display_ci_high"]
]

In [ ]:
ax = result.forest(
    effect_label="Risk ratio",
    show_prediction_interval=True,
)
ax.figure.set_size_inches(8, 5)
ax

For consequential work, save the exact package version, analysis code, input data, and structured report, then independently review the statistical choices.